# Session 1: reactions take time
**A. Real-world problem.** Milk goes sour in a day on the table and in a week in the fridge.
Hydrogen peroxide in a bottle lasts months, but a drop of a catalyst (even a bit of raw liver
or manganese dioxide) makes it fizz at once. Chemical reactions have a *speed*. Today we draw
the simplest picture of it: a concentration-versus-time curve for A -> B, and we see what a
faster rate constant does to it.

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

## Step 1: the curve for A -> B
Molecules of A turn into B. The more A there is, the more often A molecules collide with
something useful, so the rate is proportional to the concentration of A:

    rate = k [A]          [A] = A0 * exp(-k t)

`k` is the *rate constant* (units 1/s). A big k means a fast reaction. B is what A has lost:
[B] = A0 - [A].

In [ ]:
def conc_A(t, A0, k):
    """Concentration of A (mol/L) at time t (s) for A -> B with rate constant k (1/s)."""
    return A0 * np.exp(-k * np.asarray(t, dtype=float))


def conc_B(t, A0, k):
    return A0 - conc_A(t, A0, k)


def half_life(k):
    """Time (s) for half of A to react. It does not depend on how much A you start with."""
    return float(np.log(2.0) / k)


def time_to_fraction(k, fraction):
    """Time (s) until the given fraction (0 to 1, exclusive of 1) of A has turned into B."""
    return float(np.log(1.0 / (1.0 - fraction)) / k)

## Step 2: a catalyst and a hotter pot change k
A catalyst (or a higher temperature) does not change what the reaction ends with; it changes
k. We model "a catalyst that makes the step f times faster" simply as k -> f * k.

In [ ]:
def with_catalyst(k, factor):
    return k * factor

## Step 3: why the curve looks like that - count molecules
Take 2000 molecules. In each tiny time step every remaining A molecule has the same small
chance p of reacting (a "lucky collision"). Nobody tells the molecules to slow down; the curve
bends because fewer A molecules are left to react.

In [ ]:
def count_molecules(n=2000, p=0.05, steps=60, seed=1):
    """Number of A molecules left after each step (index 0 = start). p = chance per step."""
    rng = np.random.default_rng(seed)
    left = [n]
    for _ in range(steps):
        reacted = rng.random(left[-1]) < p
        left.append(left[-1] - int(reacted.sum()))
    return np.array(left)

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    t = np.linspace(0, 40, 400)
    fig, ax = plt.subplots(figsize=(6.4, 4))
    for f, ls in ((1, "-"), (5, "--")):
        k = with_catalyst(0.10, f)
        ax.plot(t, conc_A(t, 1.0, k), ls, color="tab:blue", label=f"[A], k = {k:.2f} 1/s")
        ax.plot(t, conc_B(t, 1.0, k), ls, color="tab:orange", label=f"[B], k = {k:.2f} 1/s")
    ax.axhline(0.5, color="gray", ls=":")
    ax.set_xlabel("time (s)")
    ax.set_ylabel("concentration (mol/L)")
    ax.set_title("A -> B: a 5 times larger k finishes 5 times sooner")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(OUT / "a_to_b.png", dpi=120)

    steps = 60
    counts = count_molecules(2000, 0.05, steps)
    fig2, ax2 = plt.subplots(figsize=(6.4, 4))
    ax2.plot(range(steps + 1), counts / 2000, "o", ms=3, label="counted (2000 molecules)")
    ax2.plot(range(steps + 1), 0.95 ** np.arange(steps + 1), "-", label="(1 - p)^n, the smooth curve")
    ax2.set_xlabel("time step (each step is a short time)")
    ax2.set_ylabel("fraction of A left (no unit)")
    ax2.set_title("Random collisions give the smooth exponential curve")
    ax2.grid(alpha=0.3)
    ax2.legend(fontsize=8)
    fig2.tight_layout()
    fig2.savefig(OUT / "molecule_count.png", dpi=120)

    k = 0.10
    print(f"k = {k} 1/s: [A] at 10 s = {float(conc_A(10, 1.0, k)):.3f} mol/L (A0 = 1.0)")
    print(f"half-life = {half_life(k):.2f} s;  90% converted after {time_to_fraction(k, 0.9):.1f} s")
    k5 = with_catalyst(k, 5)
    print(f"with a 5x catalyst: half-life {half_life(k5):.2f} s;  90% after {time_to_fraction(k5, 0.9):.1f} s")
    print("counted fraction left after 60 steps:", counts[-1] / 2000, " smooth:", round(0.95 ** 60, 4))

## Where this stops
Here k was just a number we typed in. Session 2 asks where k comes from (temperature, activation
energy, catalyst) and what happens if the rate depends on [A] differently.

## Coding tasks
1. **Easy.** Change `k` and `A0` in `lesson.py`. Predict [A] after 10 s for k = 0.2 and A0 = 3.0 first, then run it.
2. **Stretch.** Write `time_to_remaining(k, left)` that returns the time for [A] to fall to a fraction `left` of A0, and use it to find when 1 percent of A is left for k = 0.10 1/s.
3. **Explain.** Explain, in your own words, why the molecule-counting experiment follows (1 - p)^n and why that is the same shape as exp(-k t).

## Assessment
- **A. Conceptual.** Why does the concentration of A fall quickly at first and slowly later, even though nothing about the reaction changes?
- **B. Numerical.** A reaction A -> B has k = 0.05 1/s and A0 = 2.0 mol/L. Find [A] after 20 s and the half-life, and check your answer for sense.
- **C. Graph and simulation interpretation.** On the A-B graph of the slider app (A0 = 1, k = 0.10), where do the curves cross and what value do they have there? What would change if A0 were 2?
- **D. Design and optimisation challenge.** Design: a reaction with k = 0.10 1/s must be 90 percent complete within 10 s. By what factor must a catalyst raise k? Which whole-number factor would you choose, and what other things besides speed matter?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    checks.append(("at t = 0 all of A is still there", abs(float(L.conc_A(0.0, 1.0, 0.1)) - 1.0) < 1e-12))
    checks.append(("A decreases with time", bool(np.all(np.diff(L.conc_A(np.linspace(0, 30, 50), 1.0, 0.1)) < 0))))
    t = np.linspace(0, 50, 7)
    checks.append(("A + B is always A0 (mass is conserved)", np.allclose(L.conc_A(t, 2.0, 0.07) + L.conc_B(t, 2.0, 0.07), 2.0)))
    a10 = float(L.conc_A(10.0, 1.0, 0.10))
    checks.append((f"worked example: [A] after 10 s with k = 0.10 1/s is exp(-1) = 0.368 (got {a10:.3f})", abs(a10 - np.exp(-1)) < 1e-9))
    checks.append(("half-life for k = 0.10 1/s is ln 2 / 0.10 = 6.93 s", abs(L.half_life(0.10) - 6.9315) < 1e-3))
    checks.append(("at the half-life [A] is A0/2", abs(float(L.conc_A(L.half_life(0.3), 4.0, 0.3)) - 2.0) < 1e-9))
    checks.append(("hand calculation: 90 percent takes ln(10)/k = 23.0 s", abs(L.time_to_fraction(0.10, 0.9) - 23.026) < 1e-2))
    checks.append(("a 5 times faster catalyst finishes 5 times sooner", abs(L.time_to_fraction(L.with_catalyst(0.1, 5), 0.9) * 5 - L.time_to_fraction(0.1, 0.9)) < 1e-9))
    checks.append(("limit: k = 0 means nothing reacts", abs(float(L.conc_A(100.0, 1.0, 0.0)) - 1.0) < 1e-12))
    counts = L.count_molecules(2000, 0.05, 60)
    checks.append(("counting molecules agrees with (1 - p)^n within 3 percent", abs(counts[-1] / 2000 - 0.95 ** 60) < 0.03))
    checks.append(("the molecule count never increases", bool(np.all(np.diff(counts) <= 0))))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
